# Week 2, day 5 (morning) — Extra practice 10 SOLUTIONS: function arguments   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q4 is the one to run twice and stare at.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 10 — Function arguments. Run this once.
scores = [88, 42, 95, 61, 77]
settings = {"host": "localhost", "port": 5432, "user": "lab"}
extra = {"timeout": 30, "retries": 3}

print("scores:  ", scores)
print("settings:", settings)

### Question 1

Defaults. -> `host=db1 port=5432 secure=True`; `port=6432`; `secure=False`; then both changed.

Four calls, one definition. The defaults mean the common case is one
argument and every variation is one keyword away.

The third call — `connect("db1", secure=False)` — skips over `port`
entirely, which is only possible **by name**. Positionally you would have
had to restate the default port just to reach the argument after it.

In [ ]:
def connect(host, port=5432, secure=True):
    print(f"host={host} port={port} secure={secure}")

connect("db1")
connect("db1", 6432)
connect("db1", secure=False)
connect(host="db1", port=6432, secure=False)

### Question 2

Keyword-only, on purpose. -> `host=db1 port=6432 secure=True`, then both changed.

The bare `*` takes no arguments and collects nothing. It is a marker:
everything after it must be passed by name.

`connect_strict("db1", 6432)` raises `TypeError: takes 1 positional
argument but 2 were given`. That is the point — worksheet 10 Q10 asked you
to pass options by name as a matter of discipline, and this makes it a rule
the language enforces.

It also frees you to reorder or insert keyword-only parameters later
without silently changing the meaning of anyone's existing call.

In [ ]:
def connect_strict(host, *, port=5432, secure=True):
    print(f"host={host} port={port} secure={secure}")

connect_strict("db1", port=6432)
connect_strict("db1", port=6432, secure=False)

# connect_strict("db1", 6432) would raise TypeError: connect_strict() takes 1
# positional argument but 2 were given. That is the point -- it makes the
# readability rule from worksheet 10 Q10 impossible to break.

### Question 3

`*args`, including empty. -> `three: 3 values, total 6`; **`none: 0 values, total 0`**; `scores: 5 values, total 363`.

`*values` accepts **zero or more**, so calling with nothing is legal:
`values` is the empty tuple, `len` is 0, and `sum(())` is 0.

That is convenient and it is a trap. This function reports a total of 0
for "no data" and would report 0 for "data that sums to zero" — the same
output for two very different situations. If the difference matters, check
`len(values)` first.

`*scores` at the call site spreads the list into five arguments, as in
worksheet 10 Q5.

In [ ]:
def report(label, *values):
    print(f"{label}: {len(values)} values, total {sum(values)}")

report("three", 1, 2, 3)
report("none")
report("scores", *scores)

### Question 4

The mutable default, dictionary edition. -> `{'a': 1}`, **`{'a': 1, 'b': 2}`**, **`{'a': 1, 'b': 2, 'c': 3}`**, **`{'a': 99, 'b': 2, 'c': 3}`** — then the fixed version gives `{'a': 1}` and `{'b': 2}`.

One dictionary, created once when the `def` line ran, shared by all four
calls. The fourth call did not even add an entry — it **overwrote**
`a`, which is the same accumulation wearing a disguise.

This is harder to spot than the list version in worksheet 10 Q4, because a
dict that keeps its size looks less obviously wrong than a list that keeps
growing. A cache that never resets, a config that inherits the last
caller's overrides, a "fresh" record that is not.

Same fix, always: `store=None`, and build the real default inside the body
where it runs on **every** call.

In [ ]:
def remember(key, value, store={}):
    store[key] = value
    return store

print(remember("a", 1))
print(remember("b", 2))
print(remember("c", 3))
print(remember("a", 99))

def remember_safe(key, value, store=None):
    if store is None:
        store = {}
    store[key] = value
    return store

print(remember_safe("a", 1))
print(remember_safe("b", 2))

### Question 5

`**kwargs`, four ways. -> `{'debug': True, 'verbose': False}` / `2`; `{}` / `0`; the three settings / `3`; all five merged / `5`.

`**kwargs` with nothing passed is an empty dict, not `None` — so
`for k, v in options.items()` is safe without a guard.

`configure(**settings, **extra)` merges two dictionaries at the call site.
It works because their keys do not overlap; a shared key raises `TypeError:
got multiple values for keyword argument`, which is Q9's error arriving
from a different direction.

Note what `**` costs you: inside the function there is no signature saying
which options are valid, so a typo like `verbse=True` is accepted silently
and simply ignored.

In [ ]:
def configure(**options):
    print(options)
    return len(options)

print(configure(debug=True, verbose=False))
print(configure())
print(configure(**settings))
print(configure(**settings, **extra))

### Question 6

A default built inside the body. -> `88 B`, `42 F`, `95 A`, `61 F`, `77 C`, then `95 pass`.

`bands=None` then building the list inside is Q4's fix applied
prophylactically — the default is a list, so it must not sit in the
signature.

It also makes the function configurable: the last call passes
`[(50, "pass")]` and gets a completely different scheme with no change to
the code. That is worksheet 14 Q6's idea in a simpler form — the caller
supplies the policy, the function supplies the loop.

The bands must be in descending order for the `return` on first match to be
correct. That is worksheet 01 Q5's ordering bug waiting to happen, and it
is now the **caller's** responsibility — which is worth a docstring.

In [ ]:
def grade(score, bands=None):
    if bands is None:
        bands = [(90, "A"), (80, "B"), (70, "C")]
    for threshold, letter in bands:
        if score >= threshold:
            return letter
    return "F"

for s in scores:
    print(s, grade(s))

print(95, grade(95, [(50, "pass")]))

### Question 7

Ordering. -> correct call: `tags=('urgent', 'q3', 'draft') owner=ana meta={'team': 'data', 'version': 2}`. Second call: **`tags=('ana', 'urgent', 'q3') owner=nobody meta={}`**.

`"ana"` was meant to be the owner and became the first tag; `owner` fell
back to `"nobody"`. No error.

This is worksheet 10 Q8 exactly. Once `*tags` is in the signature there is
no position left for `owner` — it is keyword-only, and anything positional
after the name is a tag by definition.

The function cannot distinguish a missing owner from an intended one. If
`owner` really is required, give it no default and let the `TypeError`
protect you.

In [ ]:
def tag(name, *tags, owner="nobody", **meta):
    print(f"name={name} tags={tags} owner={owner} meta={meta}")

tag("report", "urgent", "q3", "draft", owner="ana", team="data", version=2)
tag("report", "ana", "urgent", "q3")

### Question 8

Introspecting the signature. -> `tag | defaults: None | kwdefaults: {'owner': 'nobody'}`; `connect | defaults: (5432, True) | kwdefaults: None`; `grade | defaults: (None,) | kwdefaults: None`.

`__defaults__` holds the defaults of **positional** parameters, as a tuple;
`__kwdefaults__` holds the keyword-only ones, as a dict. `tag` has no
positional defaults at all — `owner` is keyword-only, which is exactly what
Q7 discovered the hard way.

`grade`'s default is `(None,)`, which is the visible trace of Q6's fix: the
real default list is not in the signature at all, it is built in the body.

These are debugging tools, not something to write into a program. But when
a call is not doing what you expect, printing them answers "what does this
function actually accept" faster than reading it.

In [ ]:
for func in [tag, connect, grade]:
    print(func.__name__,
          "| defaults:", func.__defaults__,
          "| kwdefaults:", func.__kwdefaults__)

### Question 9

Filling the same parameter twice. -> a working call prints, then `TypeError: connect() got multiple values for argument 'host'`.

The first argument filled `host` by position; then `host="db2"` tried to
fill it again. Python refuses to choose, and does not silently prefer
either one.

It looks like Q1's fourth call, which named everything — the difference is
that there, nothing was passed positionally, so there was no conflict.

In real code this almost always arrives through `**` unpacking: a
dictionary that happens to contain `host`, splatted into a call that
already supplied a host positionally. The fix is to pass everything by
keyword, or everything by position, and not to mix the two for the same
parameter.

In [ ]:
connect("db1", 6432)

# This is SUPPOSED to raise: TypeError: connect() got multiple values for
# argument 'host'.
#
# It looks like Q1's fourth call, which named everything -- but here the
# first argument already filled `host` by position, and then a keyword tried
# to fill it again. Python will not choose between them.
#
# The same error appears when a **dict being unpacked contains a key that a
# positional argument has already supplied, which is how it usually turns up
# in real code.
connect("db1", host="db2")